In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# Load clean feature matrix
df = pd.read_csv('../data/feature_matrix_clean.csv')
print(f"Loaded: {df.shape}")
print(f"Target balance: {df['had_fire'].value_counts().to_dict()}")

Loaded: (10380, 46)
Target balance: {0: 9500, 1: 880}


In [2]:
# === Train/Test Split: train on 2018-2020, test on 2021 ===
# This mirrors the competition setup: use historical data to predict next year

# Feature columns (everything except zip, Year, target)
feature_cols = [col for col in df.columns if col not in ['zip', 'Year', 'had_fire']]
print(f"Number of features: {len(feature_cols)}")

# Split by year
train = df[df['Year'].isin([2018, 2019, 2020])]
test = df[df['Year'] == 2021]

X_train = train[feature_cols]
y_train = train['had_fire']
X_test = test[feature_cols]
y_test = test['had_fire']

print(f"\nTrain: {X_train.shape} | Fire rate: {y_train.mean():.2%}")
print(f"Test:  {X_test.shape} | Fire rate: {y_test.mean():.2%}")

# Scale features (important for quantum model later, good practice for all)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

print("\nScaling done.")

Number of features: 43

Train: (7786, 43) | Fire rate: 8.43%
Test:  (2594, 43) | Fire rate: 8.64%

Scaling done.


In [3]:
# === Evaluation helper ===
def evaluate_model(name, y_true, y_pred, y_prob=None):
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")
    if y_prob is not None:
        print(f"AUC-ROC:   {roc_auc_score(y_true, y_prob):.4f}")
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    print(f"  TN={cm[0][0]:>4d}  FP={cm[0][1]:>4d}")
    print(f"  FN={cm[1][0]:>4d}  TP={cm[1][1]:>4d}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['No Fire', 'Fire'])}")

In [4]:
# === Model 1: Random Forest ===
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',  # handles imbalanced classes
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

rf_pred = rf.predict(X_test_scaled)
rf_prob = rf.predict_proba(X_test_scaled)[:, 1]

evaluate_model("Random Forest", y_test, rf_pred, rf_prob)


  Random Forest
Accuracy:  0.8601
Precision: 0.3505
Recall:    0.7277
F1 Score:  0.4731
AUC-ROC:   0.8987

Confusion Matrix:
  TN=2068  FP= 302
  FN=  61  TP= 163

              precision    recall  f1-score   support

     No Fire       0.97      0.87      0.92      2370
        Fire       0.35      0.73      0.47       224

    accuracy                           0.86      2594
   macro avg       0.66      0.80      0.70      2594
weighted avg       0.92      0.86      0.88      2594



In [5]:
# === Model 2: XGBoost ===
from xgboost import XGBClassifier

# Calculate scale_pos_weight for imbalanced data
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_ratio = neg_count / pos_count

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_ratio,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
xgb.fit(X_train_scaled, y_train)

xgb_pred = xgb.predict(X_test_scaled)
xgb_prob = xgb.predict_proba(X_test_scaled)[:, 1]

evaluate_model("XGBoost", y_test, xgb_pred, xgb_prob)


  XGBoost
Accuracy:  0.8890
Precision: 0.4116
Recall:    0.6652
F1 Score:  0.5085
AUC-ROC:   0.9050

Confusion Matrix:
  TN=2157  FP= 213
  FN=  75  TP= 149

              precision    recall  f1-score   support

     No Fire       0.97      0.91      0.94      2370
        Fire       0.41      0.67      0.51       224

    accuracy                           0.89      2594
   macro avg       0.69      0.79      0.72      2594
weighted avg       0.92      0.89      0.90      2594



In [6]:
# === Feature Importance (XGBoost) ===
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Top 15 Most Important Features (XGBoost) ===")
for i, row in importance.head(15).iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"  {row['feature']:45s} {row['importance']:.4f}  {bar}")

=== Top 15 Most Important Features (XGBoost) ===
  Number of High Fire Risk Exposure             0.1623  ████████████████
  prior_max_acres                               0.0600  ██████
  Avg PPC                                       0.0556  █████
  Earned Premium                                0.0495  ████
  min_precip                                    0.0482  ████
  CAT Cov A Fire -  Number of Claims            0.0412  ████
  prior_total_acres                             0.0411  ████
  renter_occupied_housing_units                 0.0315  ███
  educational_attainment_bachelor_or_higher     0.0275  ██
  housing_vacancy_number                        0.0218  ██
  Number of Low Fire Risk Exposure              0.0212  ██
  fire_season_total_precip                      0.0210  ██
  fire_season_mean_tmax                         0.0197  █
  mean_tmax                                     0.0184  █
  Number of Very High Fire Risk Exposure        0.0178  █


In [7]:
# === Reduced feature set for quantum model comparison ===
# Select top features based on importance + what PCA will capture well
# We will use PCA to reduce to 6 components (6 qubits in the quantum circuit)

from sklearn.decomposition import PCA

# First, let's see how much variance PCA captures at different component counts
pca_full = PCA(n_components=min(43, X_train_scaled.shape[1]))
pca_full.fit(X_train_scaled)

cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)
for n in [4, 6, 8, 10, 12, 15, 20]:
    print(f"  PCA with {n:>2d} components: {cumulative_var[n-1]:.2%} variance explained")

  PCA with  4 components: 54.72% variance explained
  PCA with  6 components: 66.38% variance explained
  PCA with  8 components: 74.10% variance explained
  PCA with 10 components: 80.77% variance explained
  PCA with 12 components: 85.85% variance explained
  PCA with 15 components: 90.92% variance explained
  PCA with 20 components: 95.64% variance explained


In [8]:
# === Build PCA-reduced feature sets ===
N_COMPONENTS = 6  # matches qubit count for quantum model

pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA reduced: {X_train_scaled.shape[1]} features -> {N_COMPONENTS} components")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# === Random Forest on reduced features ===
rf_reduced = RandomForestClassifier(
    n_estimators=200, max_depth=10, 
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_reduced.fit(X_train_pca, y_train)
rf_red_pred = rf_reduced.predict(X_test_pca)
rf_red_prob = rf_reduced.predict_proba(X_test_pca)[:, 1]
evaluate_model("Random Forest (6 PCA features)", y_test, rf_red_pred, rf_red_prob)

# === XGBoost on reduced features ===
xgb_reduced = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_ratio, random_state=42,
    eval_metric='logloss', n_jobs=-1
)
xgb_reduced.fit(X_train_pca, y_train)
xgb_red_pred = xgb_reduced.predict(X_test_pca)
xgb_red_prob = xgb_reduced.predict_proba(X_test_pca)[:, 1]
evaluate_model("XGBoost (6 PCA features)", y_test, xgb_red_pred, xgb_red_prob)

PCA reduced: 43 features -> 6 components
Variance explained: 66.38%

  Random Forest (6 PCA features)
Accuracy:  0.8524
Precision: 0.3268
Recall:    0.6696
F1 Score:  0.4392
AUC-ROC:   0.8846

Confusion Matrix:
  TN=2061  FP= 309
  FN=  74  TP= 150

              precision    recall  f1-score   support

     No Fire       0.97      0.87      0.91      2370
        Fire       0.33      0.67      0.44       224

    accuracy                           0.85      2594
   macro avg       0.65      0.77      0.68      2594
weighted avg       0.91      0.85      0.87      2594


  XGBoost (6 PCA features)
Accuracy:  0.8466
Precision: 0.3125
Recall:    0.6473
F1 Score:  0.4215
AUC-ROC:   0.8672

Confusion Matrix:
  TN=2051  FP= 319
  FN=  79  TP= 145

              precision    recall  f1-score   support

     No Fire       0.96      0.87      0.91      2370
        Fire       0.31      0.65      0.42       224

    accuracy                           0.85      2594
   macro avg       0.64      

In [9]:
# === Save PCA-reduced data for quantum model notebook ===
import pickle

np.save('../data/X_train_pca.npy', X_train_pca)
np.save('../data/X_test_pca.npy', X_test_pca)
np.save('../data/y_train.npy', y_train.values)
np.save('../data/y_test.npy', y_test.values)

# Save scaler and PCA for later use
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../data/pca.pkl', 'wb') as f:
    pickle.dump(pca, f)

print("Saved PCA-reduced data for quantum model.")
print(f"  X_train_pca: {X_train_pca.shape}")
print(f"  X_test_pca: {X_test_pca.shape}")
print(f"  y_train: {y_train.shape}, fire rate: {y_train.mean():.2%}")
print(f"  y_test: {y_test.shape}, fire rate: {y_test.mean():.2%}")

# === Summary comparison so far ===
print(f"\n{'='*60}")
print(f"  COMPARISON SUMMARY")
print(f"{'='*60}")
print(f"{'Model':<35s} {'AUC':>6s} {'F1':>6s} {'Recall':>7s} {'Prec':>6s}")
print(f"{'-'*60}")
print(f"{'RF (43 features)':<35s} {roc_auc_score(y_test, rf_prob):>6.4f} {f1_score(y_test, rf_pred):>6.4f} {recall_score(y_test, rf_pred):>7.4f} {precision_score(y_test, rf_pred):>6.4f}")
print(f"{'XGB (43 features)':<35s} {roc_auc_score(y_test, xgb_prob):>6.4f} {f1_score(y_test, xgb_pred):>6.4f} {recall_score(y_test, xgb_pred):>7.4f} {precision_score(y_test, xgb_pred):>6.4f}")
print(f"{'RF (6 PCA features)':<35s} {roc_auc_score(y_test, rf_red_prob):>6.4f} {f1_score(y_test, rf_red_pred):>6.4f} {recall_score(y_test, rf_red_pred):>7.4f} {precision_score(y_test, rf_red_pred):>6.4f}")
print(f"{'XGB (6 PCA features)':<35s} {roc_auc_score(y_test, xgb_red_prob):>6.4f} {f1_score(y_test, xgb_red_pred):>6.4f} {recall_score(y_test, xgb_red_pred):>7.4f} {precision_score(y_test, xgb_red_pred):>6.4f}")
print(f"{'Quantum VQC (6 PCA features)':<35s} {'TBD':>6s} {'TBD':>6s} {'TBD':>7s} {'TBD':>6s}")

Saved PCA-reduced data for quantum model.
  X_train_pca: (7786, 6)
  X_test_pca: (2594, 6)
  y_train: (7786,), fire rate: 8.43%
  y_test: (2594,), fire rate: 8.64%

  COMPARISON SUMMARY
Model                                  AUC     F1  Recall   Prec
------------------------------------------------------------
RF (43 features)                    0.8987 0.4731  0.7277 0.3505
XGB (43 features)                   0.9050 0.5085  0.6652 0.4116
RF (6 PCA features)                 0.8846 0.4392  0.6696 0.3268
XGB (6 PCA features)                0.8672 0.4215  0.6473 0.3125
Quantum VQC (6 PCA features)           TBD    TBD     TBD    TBD


In [10]:
# === Validation on 2022 and 2023 (partial evaluation) ===
# We can only check: do the zip codes that actually had fires get high risk scores?

# Retrain on ALL of 2018-2021 for this check
X_all = df[feature_cols]
y_all = df['had_fire']
scaler_full = StandardScaler()
X_all_scaled = pd.DataFrame(scaler_full.fit_transform(X_all), columns=feature_cols)

xgb_full = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=(y_all==0).sum()/(y_all==1).sum(),
    random_state=42, eval_metric='logloss', n_jobs=-1
)
xgb_full.fit(X_all_scaled, y_all)

# Get predictions for all zip codes using their most recent features (2021)
latest = df[df['Year'] == 2021].copy()
X_latest = latest[feature_cols]
X_latest_scaled = pd.DataFrame(scaler_full.transform(X_latest), columns=feature_cols)
latest['fire_prob'] = xgb_full.predict_proba(X_latest_scaled)[:, 1]

# Load wildfire data to get actual 2022 and 2023 fire zip codes
wildfire = pd.read_csv('../data/wildfire_weather.csv', low_memory=False)
fire_zips_2022 = set(wildfire[(wildfire['FIRE_NAME'].notna()) & (wildfire['Year'] == 2022)]['zip'].dropna().astype(int))
fire_zips_2023 = set(wildfire[(wildfire['FIRE_NAME'].notna()) & (wildfire['Year'] == 2023)]['zip'].dropna().astype(int))

print(f"Known fire zip codes: 2022={len(fire_zips_2022)}, 2023={len(fire_zips_2023)}")

# Check how our model ranks these zip codes
for year_label, fire_zips in [('2022', fire_zips_2022), ('2023', fire_zips_2023)]:
    latest_copy = latest.copy()
    latest_copy['actual_fire'] = latest_copy['zip'].astype(int).isin(fire_zips).astype(int)
    
    matched = latest_copy[latest_copy['actual_fire'] == 1]
    not_matched = latest_copy[latest_copy['actual_fire'] == 0]
    
    print(f"\n{'='*50}")
    print(f"  {year_label} Validation")
    print(f"{'='*50}")
    print(f"Fire zips found in our prediction set: {len(matched)} / {len(fire_zips)}")
    print(f"Avg predicted fire probability:")
    print(f"  Actual fire zips:    {matched['fire_prob'].mean():.4f}")
    print(f"  Non-fire zips:       {not_matched['fire_prob'].mean():.4f}")
    print(f"  Ratio:               {matched['fire_prob'].mean() / not_matched['fire_prob'].mean():.1f}x higher")
    
    # What % of actual fire zips are in the model's top-N riskiest?
    latest_sorted = latest_copy.sort_values('fire_prob', ascending=False)
    for top_n in [100, 200, 300, 500]:
        top_zips = set(latest_sorted.head(top_n)['zip'].astype(int))
        caught = len(fire_zips & top_zips)
        print(f"  Top {top_n} riskiest zips capture {caught}/{len(matched)} ({caught/max(len(matched),1):.0%}) of actual fires")

Known fire zip codes: 2022=173, 2023=119

  2022 Validation
Fire zips found in our prediction set: 173 / 173
Avg predicted fire probability:
  Actual fire zips:    0.5200
  Non-fire zips:       0.1128
  Ratio:               4.6x higher
  Top 100 riskiest zips capture 40/173 (23%) of actual fires
  Top 200 riskiest zips capture 67/173 (39%) of actual fires
  Top 300 riskiest zips capture 86/173 (50%) of actual fires
  Top 500 riskiest zips capture 118/173 (68%) of actual fires

  2023 Validation
Fire zips found in our prediction set: 118 / 119
Avg predicted fire probability:
  Actual fire zips:    0.5670
  Non-fire zips:       0.1196
  Ratio:               4.7x higher
  Top 100 riskiest zips capture 32/118 (27%) of actual fires
  Top 200 riskiest zips capture 56/118 (47%) of actual fires
  Top 300 riskiest zips capture 64/118 (54%) of actual fires
  Top 500 riskiest zips capture 82/118 (69%) of actual fires


In [11]:
# === Risk score distribution ===
print("=== Predicted Fire Probability Distribution ===")
print(latest['fire_prob'].describe().round(4))

print(f"\n=== Risk Tiers ===")
for threshold in [0.1, 0.2, 0.3, 0.5]:
    flagged = (latest['fire_prob'] >= threshold).sum()
    print(f"  Prob >= {threshold}: {flagged} zip codes flagged ({flagged/len(latest):.1%})")

# How well does the model separate fire from no-fire for 2021 itself?
latest_2021 = latest.copy()
latest_2021['actual_fire_2021'] = y_all[latest.index].values

print(f"\n=== 2021 Sanity Check (training data, expect good separation) ===")
fire_21 = latest_2021[latest_2021['actual_fire_2021'] == 1]['fire_prob']
nofire_21 = latest_2021[latest_2021['actual_fire_2021'] == 0]['fire_prob']
print(f"  Fire zips avg prob:    {fire_21.mean():.4f} (median: {fire_21.median():.4f})")
print(f"  No-fire zips avg prob: {nofire_21.mean():.4f} (median: {nofire_21.median():.4f})")

=== Predicted Fire Probability Distribution ===
count    2594.0000
mean        0.1400
std         0.2727
min         0.0000
25%         0.0006
50%         0.0116
75%         0.1172
max         0.9999
Name: fire_prob, dtype: float64

=== Risk Tiers ===
  Prob >= 0.1: 691 zip codes flagged (26.6%)
  Prob >= 0.2: 481 zip codes flagged (18.5%)
  Prob >= 0.3: 373 zip codes flagged (14.4%)
  Prob >= 0.5: 259 zip codes flagged (10.0%)

=== 2021 Sanity Check (training data, expect good separation) ===
  Fire zips avg prob:    0.9455 (median: 0.9594)
  No-fire zips avg prob: 0.0639 (median: 0.0066)


In [12]:
# === Same check with PCA-reduced model (what quantum will use) ===
pca_full_check = PCA(n_components=6, random_state=42)
X_all_pca = pca_full_check.fit_transform(X_all_scaled)

xgb_pca_full = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=(y_all==0).sum()/(y_all==1).sum(),
    random_state=42, eval_metric='logloss', n_jobs=-1
)
xgb_pca_full.fit(X_all_pca, y_all)

X_latest_pca = pca_full_check.transform(X_latest_scaled)
latest['fire_prob_pca'] = xgb_pca_full.predict_proba(X_latest_pca)[:, 1]

for year_label, fire_zips in [('2022', fire_zips_2022), ('2023', fire_zips_2023)]:
    latest_copy = latest.copy()
    latest_copy['actual_fire'] = latest_copy['zip'].astype(int).isin(fire_zips).astype(int)
    matched = latest_copy[latest_copy['actual_fire'] == 1]
    not_matched = latest_copy[latest_copy['actual_fire'] == 0]
    
    print(f"\n=== {year_label} Validation (PCA-6 model) ===")
    print(f"Fire zips found: {len(matched)} / {len(fire_zips)}")
    print(f"Avg predicted probability:")
    print(f"  Actual fire zips: {matched['fire_prob_pca'].mean():.4f}")
    print(f"  Non-fire zips:    {not_matched['fire_prob_pca'].mean():.4f}")
    print(f"  Ratio:            {matched['fire_prob_pca'].mean() / not_matched['fire_prob_pca'].mean():.1f}x higher")
    
    latest_sorted = latest_copy.sort_values('fire_prob_pca', ascending=False)
    for top_n in [100, 200, 300, 500]:
        top_zips = set(latest_sorted.head(top_n)['zip'].astype(int))
        caught = len(fire_zips & top_zips)
        print(f"  Top {top_n} riskiest capture {caught}/{len(matched)} ({caught/max(len(matched),1):.0%}) of fires")


=== 2022 Validation (PCA-6 model) ===
Fire zips found: 173 / 173
Avg predicted probability:
  Actual fire zips: 0.6253
  Non-fire zips:    0.1705
  Ratio:            3.7x higher
  Top 100 riskiest capture 43/173 (25%) of fires
  Top 200 riskiest capture 73/173 (42%) of fires
  Top 300 riskiest capture 90/173 (52%) of fires
  Top 500 riskiest capture 117/173 (68%) of fires

=== 2023 Validation (PCA-6 model) ===
Fire zips found: 118 / 119
Avg predicted probability:
  Actual fire zips: 0.6680
  Non-fire zips:    0.1786
  Ratio:            3.7x higher
  Top 100 riskiest capture 37/118 (31%) of fires
  Top 200 riskiest capture 57/118 (48%) of fires
  Top 300 riskiest capture 65/118 (55%) of fires
  Top 500 riskiest capture 87/118 (74%) of fires


In [13]:
# === PCA with 10 components ===
pca10 = PCA(n_components=10, random_state=42)
X_train_pca10 = pca10.fit_transform(X_train_scaled)
X_test_pca10 = pca10.transform(X_test_scaled)

print(f"PCA 10 components: {pca10.explained_variance_ratio_.sum():.2%} variance explained")

# Classical baselines on 10 PCA features
rf_10 = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_10.fit(X_train_pca10, y_train)
rf_10_pred = rf_10.predict(X_test_pca10)
rf_10_prob = rf_10.predict_proba(X_test_pca10)[:, 1]
evaluate_model("Random Forest (10 PCA features)", y_test, rf_10_pred, rf_10_prob)

xgb_10 = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_ratio, random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_10.fit(X_train_pca10, y_train)
xgb_10_pred = xgb_10.predict(X_test_pca10)
xgb_10_prob = xgb_10.predict_proba(X_test_pca10)[:, 1]
evaluate_model("XGBoost (10 PCA features)", y_test, xgb_10_pred, xgb_10_prob)

# Save for quantum notebook
np.save('../data/X_train_pca10.npy', X_train_pca10)
np.save('../data/X_test_pca10.npy', X_test_pca10)

import pickle
with open('../data/pca10.pkl', 'wb') as f:
    pickle.dump(pca10, f)

print(f"\nSaved 10-component PCA data.")
print(f"X_train_pca10: {X_train_pca10.shape}")
print(f"X_test_pca10: {X_test_pca10.shape}")

PCA 10 components: 80.77% variance explained

  Random Forest (10 PCA features)
Accuracy:  0.8604
Precision: 0.3357
Recall:    0.6295
F1 Score:  0.4379
AUC-ROC:   0.8869

Confusion Matrix:
  TN=2091  FP= 279
  FN=  83  TP= 141

              precision    recall  f1-score   support

     No Fire       0.96      0.88      0.92      2370
        Fire       0.34      0.63      0.44       224

    accuracy                           0.86      2594
   macro avg       0.65      0.76      0.68      2594
weighted avg       0.91      0.86      0.88      2594


  XGBoost (10 PCA features)
Accuracy:  0.8712
Precision: 0.3568
Recall:    0.6116
F1 Score:  0.4507
AUC-ROC:   0.8712

Confusion Matrix:
  TN=2123  FP= 247
  FN=  87  TP= 137

              precision    recall  f1-score   support

     No Fire       0.96      0.90      0.93      2370
        Fire       0.36      0.61      0.45       224

    accuracy                           0.87      2594
   macro avg       0.66      0.75      0.69      2

In [14]:
# === PCA with 12 components ===
pca12 = PCA(n_components=12, random_state=42)
X_train_pca12 = pca12.fit_transform(X_train_scaled)
X_test_pca12 = pca12.transform(X_test_scaled)

print(f"PCA 12 components: {pca12.explained_variance_ratio_.sum():.2%} variance explained")

rf_12 = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_12.fit(X_train_pca12, y_train)
rf_12_pred = rf_12.predict(X_test_pca12)
rf_12_prob = rf_12.predict_proba(X_test_pca12)[:, 1]
evaluate_model("Random Forest (12 PCA features)", y_test, rf_12_pred, rf_12_prob)

xgb_12 = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_ratio, random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_12.fit(X_train_pca12, y_train)
xgb_12_pred = xgb_12.predict(X_test_pca12)
xgb_12_prob = xgb_12.predict_proba(X_test_pca12)[:, 1]
evaluate_model("XGBoost (12 PCA features)", y_test, xgb_12_pred, xgb_12_prob)

np.save('../data/X_train_pca12.npy', X_train_pca12)
np.save('../data/X_test_pca12.npy', X_test_pca12)

import pickle
with open('../data/pca12.pkl', 'wb') as f:
    pickle.dump(pca12, f)

print(f"\nSaved 12-component PCA data.")

# Updated comparison
print(f"\n{'='*70}")
print(f"  PCA COMPARISON (XGBoost)")
print(f"{'='*70}")
print(f"{'Components':<15s} {'Variance':>10s} {'AUC':>7s} {'F1':>7s} {'Recall':>7s} {'Prec':>7s}")
print(f"{'-'*70}")
for n, auc, f1, rec, prec in [
    (6,  0.8672, 0.4215, 0.6473, 0.3125),
    (10, roc_auc_score(y_test, xgb_10_prob), f1_score(y_test, xgb_10_pred), recall_score(y_test, xgb_10_pred), precision_score(y_test, xgb_10_pred)),
    (12, roc_auc_score(y_test, xgb_12_prob), f1_score(y_test, xgb_12_pred), recall_score(y_test, xgb_12_pred), precision_score(y_test, xgb_12_pred)),
    (43, 1.0, 0.5085, 0.6652, 0.4116),
]:
    var = {6: '66.4%', 10: '80.8%', 12: '85.9%', 43: '100%'}[n]
    print(f"{n:<15d} {var:>10s} {auc:>7.4f} {f1:>7.4f} {rec:>7.4f} {prec:>7.4f}")

PCA 12 components: 85.85% variance explained

  Random Forest (12 PCA features)
Accuracy:  0.8566
Precision: 0.3271
Recall:    0.6250
F1 Score:  0.4294
AUC-ROC:   0.8921

Confusion Matrix:
  TN=2082  FP= 288
  FN=  84  TP= 140

              precision    recall  f1-score   support

     No Fire       0.96      0.88      0.92      2370
        Fire       0.33      0.62      0.43       224

    accuracy                           0.86      2594
   macro avg       0.64      0.75      0.67      2594
weighted avg       0.91      0.86      0.88      2594


  XGBoost (12 PCA features)
Accuracy:  0.8751
Precision: 0.3677
Recall:    0.6205
F1 Score:  0.4618
AUC-ROC:   0.8751

Confusion Matrix:
  TN=2131  FP= 239
  FN=  85  TP= 139

              precision    recall  f1-score   support

     No Fire       0.96      0.90      0.93      2370
        Fire       0.37      0.62      0.46       224

    accuracy                           0.88      2594
   macro avg       0.66      0.76      0.70      2

In [15]:
# === Top-K feature selection (no PCA) ===
# Use the top features from XGBoost importance ranking

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

for k in [6, 10]:
    top_k = importance.head(k)['feature'].tolist()
    print(f"\n=== Top {k} features ===")
    for f in top_k:
        print(f"  {f}")
    
    X_train_topk = X_train_scaled[top_k].values
    X_test_topk = X_test_scaled[top_k].values
    
    xgb_topk = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_ratio, random_state=42, eval_metric='logloss', n_jobs=-1)
    xgb_topk.fit(X_train_topk, y_train)
    pred = xgb_topk.predict(X_test_topk)
    prob = xgb_topk.predict_proba(X_test_topk)[:, 1]
    
    auc = roc_auc_score(y_test, prob)
    f1 = f1_score(y_test, pred)
    rec = recall_score(y_test, pred)
    prec = precision_score(y_test, pred)
    print(f"  AUC={auc:.4f}  F1={f1:.4f}  Recall={rec:.4f}  Precision={prec:.4f}")
    
    # Save for quantum notebook
    np.save(f'../data/X_train_top{k}.npy', X_train_topk)
    np.save(f'../data/X_test_top{k}.npy', X_test_topk)
    
    # Save feature names
    with open(f'../data/top{k}_features.txt', 'w') as f:
        f.write('\n'.join(top_k))

print("\nSaved top-K feature data.")


=== Top 6 features ===
  Number of High Fire Risk Exposure
  prior_max_acres
  Avg PPC
  Earned Premium
  min_precip
  CAT Cov A Fire -  Number of Claims
  AUC=0.8858  F1=0.4332  Recall=0.7232  Precision=0.3092

=== Top 10 features ===
  Number of High Fire Risk Exposure
  prior_max_acres
  Avg PPC
  Earned Premium
  min_precip
  CAT Cov A Fire -  Number of Claims
  prior_total_acres
  renter_occupied_housing_units
  educational_attainment_bachelor_or_higher
  housing_vacancy_number
  AUC=0.9044  F1=0.4877  Recall=0.7098  Precision=0.3715

Saved top-K feature data.
